In [77]:
!pip install plotly

In [92]:
#табличка
import pandas as pd

file = '../data/Лофты посвяты.xlsx'
df = pd.read_excel(file).dropna(how='all', axis=1)
df.columns = ['Дата', 'Тип', 'Минимальная цена', 'Средняя цена', 'Ссылки']
print("✅ Загружено строк:", len(df))
df.head()

✅ Загружено строк: 28


,Дата,Тип,Минимальная цена,Средняя цена,Ссылки
0,2026-04-24,Лофт,1700,1966.666667,https://vk.com/wall-130344439_6003
1,2026-02-13,Лофт,1800,2000.000000,https://vk.com/wall-130344439_5958
2,2025-11-15,Посвят,3500,3733.333333,https://vk.com/wall-130344439_5928
3,2025-09-05,Лофт,1800,2000.000000,https://vk.com/wall-130344439_5909
4,2025-04-25,Лофт,1500,1600.000000,https://vk.com/wall-130344439_5836


In [94]:
#преобразуем дату
df['Дата'] = pd.to_datetime(df['Дата'], dayfirst=True, errors='coerce')
df['Год'] = df['Дата'].dt.year

#чистим цены
for col in ['Минимальная цена', 'Средняя цена']:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = df[col].str.replace(r'[^\d\.\-]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

#удаляем строки без года или без средней цены
df_clean = df.dropna(subset=['Год', 'Средняя цена']).copy()
df_clean['Тип'] = df_clean['Тип'].astype(str).str.strip().str.lower()

#отдельные датафреймы для лофт и посвят
loft = df_clean[df_clean['Тип'] == 'лофт']
posvyat = df_clean[df_clean['Тип'] == 'посвят']

# список годов
years = sorted(df_clean['Год'].unique().tolist())

print(f"Всего записей: {len(df_clean)}")
print(f"Лофт: {len(loft)}, Посвят: {len(posvyat)}")
print(f"Годы: {min(years)}-{max(years)}")

Всего записей: 28
Лофт: 19, Посвят: 9
Годы: 2017-2026


In [64]:
#средняя цена за все годы (все, лофт, посвят)

mean_all = df_clean['Средняя цена'].mean()
mean_loft = loft['Средняя цена'].mean() if not loft.empty else None
mean_posvyat = posvyat['Средняя цена'].mean() if not posvyat.empty else None

print("СРЕДНИЕ ЦЕНЫ ЗА ВСЕ ГОДЫ:")
print(f"  • Все мероприятия:        {mean_all:,.2f}")
if mean_loft is not None:
    print(f"  • Только «лофт»:           {mean_loft:,.2f}")
else:
    print("  • Только «лофт»:           нет данных")
if mean_posvyat is not None:
    print(f"  • Только «посвят»:         {mean_posvyat:,.2f}")


СРЕДНИЕ ЦЕНЫ ЗА ВСЕ ГОДЫ:
  • Все мероприятия:        2,054.62
  • Только «лофт»:           1,278.75
  • Только «посвят»:         3,692.59


In [65]:
#общий график: средняя цена всех мероприятий (лофт+посвят) по годам
import plotly.express as px

total_yearly = df_clean.groupby('Год')['Средняя цена'].mean().reset_index()

fig = px.line(total_yearly, x='Год', y='Средняя цена',
              title='Общая средняя цена проходки (лофт + посвят) по годам',
              markers=True, line_shape='linear')
fig.update_traces(line_color='green', line_width=3, marker_size=10)
fig.update_layout(xaxis=dict(tickmode='linear'), hovermode='x unified')
fig.show()

In [66]:
#сравнение динамики "лофт" и "посвят"
loft_yearly = loft.groupby('Год')['Средняя цена'].mean().reset_index()
loft_yearly['Тип'] = 'Лофт'
posvyat_yearly = posvyat.groupby('Год')['Средняя цена'].mean().reset_index()
posvyat_yearly['Тип'] = 'Посвят'
combined = pd.concat([loft_yearly, posvyat_yearly])

fig = px.line(combined, x='Год', y='Средняя цена', color='Тип',
              title='Сравнение динамики цен: Лофт vs Посвят',
              markers=True, line_shape='linear',
              color_discrete_map={'Лофт':'fuchsia', 'Посвят':'teal'})
fig.update_traces(line_width=3, marker_size=10)
fig.update_layout(xaxis=dict(tickmode='linear'), hovermode='x unified')
fig.show()

In [67]:
#сравнение средней цены "лофт" и "посвят"
pivot = df_clean.groupby(['Год', 'Тип'])['Средняя цена'].mean().unstack()
pivot = pivot[['лофт', 'посвят']].rename(columns={'лофт':'Лофт', 'посвят':'Посвят'})

fig = px.bar(pivot, x=pivot.index, y=['Лофт', 'Посвят'],
             title='Сравнение средней цены: Лофт и Посвят по годам',
             barmode='group', text_auto='.2f', color_discrete_sequence=['fuchsia', 'teal'])
fig.update_layout(xaxis_title='Год', yaxis_title='Средняя цена',
                  legend_title='Тип мероприятия')
fig.show()

In [68]:
#динамика средней цены "лофт" и "посвят"
yearly_stats = df_clean.groupby('Год').agg(
    Средняя_цена=('Средняя цена', 'mean'),
    Минимальная_цена=('Минимальная цена', 'mean')
).reset_index()

fig = px.line(yearly_stats, x='Год', y=['Средняя_цена', 'Минимальная_цена'],
              title='Динамика средней и минимальной цены (все мероприятия)',
              markers=True, labels={'value':'Цена', 'variable':'Показатель'},
              color_discrete_sequence=['pink', 'olive'])
fig.update_traces(line_width=2.5)
fig.show()